# Setup

In [ ]:
%pip install mistralai pandas numpy

In [ ]:
from mistralai.client import Mistral
import os
import json
import time
from datasets import load_dataset

In [ ]:
# Initialize the Mistral client
api_key = "YOUR_API_KEY_HERE"
API_KEY = os.environ.get("MISTRAL_API_KEY", api_key)
client = Mistral(api_key=API_KEY)

# Loading the Wikipedia corpus provided by HotpotQA for RAG implementation

https://huggingface.co/datasets/BeIR/hotpotqa 

This dataset contains the first paragraph of each Wikipedia article.

In [4]:
from datasets import load_dataset
corpus = load_dataset("BeIR/hotpotqa", "corpus", split="corpus")
print(len(corpus))
print(corpus[0])

5233329
{'_id': '12', 'title': 'Anarchism', 'text': 'Anarchism is a political philosophy that advocates self-governed societies based on voluntary institutions. These are often described as stateless societies, although several authors have defined them more specifically as institutions based on non-hierarchical free associations. Anarchism holds the state to be undesirable, unnecessary and harmful.'}


# Load HotpotQA Development set for full-Wikipedia setting

Students are instructed to use the development set with the full Wikipedia corpus as the primary testing set (for evaluation).

The code below loads the HotpotQA dataset (the questions used for testing the system).

In [2]:
# Path to your HotpotQA dataset file
file_path = "./hotpot_dev_fullwiki_v1.json"

# Open and load the dataset
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Print total number of entries
print(f"Total examples: {len(data)}\n")

# Print the questions
for i, item in enumerate(data[:10]):  # print first 10 questions for demo
    print(f"Question {i+1}: {item['question']}")

Total examples: 7405

Question 1: Were Scott Derrickson and Ed Wood of the same nationality?
Question 2: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Question 3: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Question 4: Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?
Question 5: The director of the romantic comedy "Big Stone Gap" is based in what New York city?
Question 6: 2014 S/S is the debut album of a South Korean boy group that was formed by who?
Question 7: Who was known by his stage name Aladin and helped organizations improve their performance as a consultant?
Question 8: The arena where the Lewiston Maineiacs played their home games can seat how many people?
Question 9: Who is older, Annie Morton or Terry Richardson?
Question 10: Are Local H and For Against both from the United States

# RAG

In [ ]:
def basic_prompt(question, model="mistral-small-latest"):
    """
    This function sends a prompt to the Mistral API and returns a response from the LLM. 
    
    Args:
            question (str): The question to ask, or prompt to send
            model (str): The model to use
    
    Returns:
        str: The model's response
    """

    messages = [{"role": "user", "content": question}]
    response = client.chat.complete(model=model, messages=messages)

    return response.choices[0].message.content

In [ ]:
def retrieve_documents(query, knowledge_base, top_k=2):
    """
    This function is responsible for getting the most relevant documents. 
    
    Simple keyword-based retrieval.
    (alternatively, try to use vector embeddings or other retrival methods)

    Args:
        query (str): User's question / prompt
        knowledge_base (list): List of documents
        top_k (int): Number of documents to retrieve

    Returns:
        list: Retrieved documents
    """
    # Simple keyword matching
    query_words = set(query.lower().split())

    scored_docs = []
    for doc in knowledge_base:
        doc_words = set(doc["content"].lower().split())
        score = len(query_words & doc_words)
        scored_docs.append((score, doc))

    # Sort by score and return top_k
    scored_docs.sort(reverse=True, key=lambda x: x[0])
    return [doc for score, doc in scored_docs[:top_k]]

In [ ]:
def rag_query(question, knowledge_base, model="mistral-small-latest"):
    """
    This function performs a RAG query, adding relevant documents as context
    for the prompt. 

    Args:
        question (str): User's question
        knowledge_base (list): Knowledge base to search

    Returns:
        str: Generated answer with context
    """
    # Step 1: Retrieve relevant documents
    retrieved_docs = retrieve_documents(question, knowledge_base)

    # Step 2: Construct context from retrieved documents
    context = "\n\n".join([doc["content"] for doc in retrieved_docs])

    # Step 3: Create prompt with context
    rag_prompt = f"""Use the following context to answer the question. If the answer cannot be found in the context, say so.
                Context:
                {context}

                Question: {question}

                Answer:"""
                # This "rag_prompt" is for demo usage, please design your prompt.

    return basic_prompt(rag_prompt, model), retrieved_docs

In [ ]:
# Example usage
question = "When was the Eiffel Tower built and why?"
print(f"Question: {question}\n")

answer, retrieved = rag_query(question, KNOWLEDGE_BASE)
print(f"Retrieved Documents:")
for doc in retrieved:
    print(f"- Doc {doc['id']}: {doc['content'][:50]}...")

print(f"\nAnswer:\n{answer}\n")

# Try another question
question2 = "What is machine learning?"
print(f"Question: {question2}\n")

answer2, retrieved2 = rag_query(question2, KNOWLEDGE_BASE)
print(f"Retrieved Documents:")
for doc in retrieved2:
    print(f"- Doc {doc['id']}: {doc['content'][:100]}...")

print(f"\nAnswer:\n{answer2}\n")